# F1 Race Analytics — EDA
**Race:** 2024 Japanese Grand Prix · Suzuka  
**Goal:** Compare Verstappen vs Piastri lap pace. Extends my IB Math IA (cornering analysis at Suzuka Turn 1).  
**CRISP-DM:** Business Understanding → Data Understanding → Data Preparation

In [ ]:
%matplotlib inline
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
fastf1.Cache.enable_cache('cache')

session = fastf1.get_session(2024, 'Japanese Grand Prix', 'R')
session.load()

print(f"Loaded: {session.event['EventName']} {session.event.year}")

## Data Overview

In [ ]:
laps = session.laps
print(laps.shape)
print(laps.columns.tolist())
laps.head()

In [ ]:
# check missing values in key columns
key_cols = ['Driver', 'LapTime', 'LapNumber', 'Compound', 'TyreLife',
            'Sector1Time', 'Sector2Time', 'Sector3Time']
print(laps[key_cols].isnull().sum())

In [ ]:
# pick_accurate removes pit laps, SC laps, formation lap
clean_laps = laps.pick_accurate()
print(f"All laps: {len(laps)} → Clean: {len(clean_laps)}")

## Filter: Verstappen vs Piastri

In [ ]:
ver_laps = clean_laps.pick_drivers('VER').copy()
pia_laps = clean_laps.pick_drivers('PIA').copy()

# convert timedelta to float seconds for analysis
ver_laps['LapTimeSec'] = ver_laps['LapTime'].dt.total_seconds()
pia_laps['LapTimeSec'] = pia_laps['LapTime'].dt.total_seconds()

print(f"VER: {len(ver_laps)} laps | PIA: {len(pia_laps)} laps")

## Descriptive Stats

In [ ]:
print("=== VER ===")
print(ver_laps['LapTimeSec'].describe().round(3))
print("\n=== PIA ===")
print(pia_laps['LapTimeSec'].describe().round(3))

gap = ver_laps['LapTimeSec'].median() - pia_laps['LapTimeSec'].median()
print(f"\nMedian gap (VER - PIA): {gap:+.3f}s")

## Chart 1 — Lap Time Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ver_laps['LapNumber'], ver_laps['LapTimeSec'],
        color='#0600EF', marker='o', markersize=3, linewidth=2, label='Verstappen')
ax.plot(pia_laps['LapNumber'], pia_laps['LapTimeSec'],
        color='#FF8000', marker='o', markersize=3, linewidth=2, label='Piastri')

ax.set_xlabel('Lap Number')
ax.set_ylabel('Lap Time (s)')
ax.set_title('2024 Japanese GP — Race Pace Trace')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/01_lap_time_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 2 — Distribution

In [ ]:
ver_plot = ver_laps[['LapTimeSec']].copy()
ver_plot['Driver'] = 'Verstappen'
pia_plot = pia_laps[['LapTimeSec']].copy()
pia_plot['Driver'] = 'Piastri'
combined = pd.concat([ver_plot, pia_plot], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=combined, x='Driver', y='LapTimeSec',
            hue='Driver',
            palette={'Verstappen': '#0600EF', 'Piastri': '#FF8000'},
            legend=False, ax=ax)

ax.set_ylabel('Lap Time (s)')
ax.set_title('Lap Time Distribution')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_lap_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 3 — Sector Times
Sector 1 at Suzuka contains the S-curves (the section I modelled in my IB IA).

In [ ]:
for df in [ver_laps, pia_laps]:
    for s in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
        df[s.replace('Time', 'Sec')] = df[s].dt.total_seconds()

sectors = ['Sector1Sec', 'Sector2Sec', 'Sector3Sec']
labels = ['S1 (Esses)', 'S2 (Hairpin)', 'S3 (130R)']

ver_med = [ver_laps[s].median() for s in sectors]
pia_med = [pia_laps[s].median() for s in sectors]

x = np.arange(3)
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, ver_med, w, label='Verstappen', color='#0600EF', alpha=0.85)
b2 = ax.bar(x + w/2, pia_med, w, label='Piastri', color='#FF8000', alpha=0.85)

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.2f}s', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Median Sector Time (s)')
ax.set_title('Sector Comparison — Japan 2024')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/03_sector_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 4 — Pace by Tire Compound

In [ ]:
ver_c = ver_laps[['LapTimeSec', 'Compound']].copy()
ver_c['Driver'] = 'Verstappen'
pia_c = pia_laps[['LapTimeSec', 'Compound']].copy()
pia_c['Driver'] = 'Piastri'
cmp_data = pd.concat([ver_c, pia_c]).dropna(subset=['Compound'])

print(cmp_data.groupby(['Driver', 'Compound']).size().unstack(fill_value=0))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=cmp_data, x='Compound', y='LapTimeSec', hue='Driver',
            palette={'Verstappen': '#0600EF', 'Piastri': '#FF8000'}, ax=ax)

ax.set_ylabel('Lap Time (s)')
ax.set_title('Pace by Tire Compound — Japan 2024')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_pace_by_compound.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print(f"VER median: {ver_laps['LapTimeSec'].median():.3f}s")
print(f"PIA median: {pia_laps['LapTimeSec'].median():.3f}s")
print(f"Gap: {gap:+.3f}s")
print(f"\nVER strategy: {ver_laps['Compound'].value_counts().to_dict()}")
print(f"PIA strategy: {pia_laps['Compound'].value_counts().to_dict()}")